# 01 · Sono, HRV e recuperação

Observação do lado **fisiológico** dos dados: o que o relógio registra fora do treino. Uma linha por
dia em `wellness_daily.csv`.

**Perguntas deste notebook**

1. O quanto dessa série dá para usar — quantos dias, quais colunas estão realmente preenchidas?
2. Sono, HRV, FC de repouso e estresse: quais são os níveis normais deste atleta e como oscilam?
3. Há tendência ao longo das semanas, ou só ruído do dia a dia?

O cruzamento com carga de treino **não** está aqui — mora em `03_load_recovery`, junto com o recorte
de janela que ele exige.

In [ ]:
from processing.datasets import load_wellness
from processing.features import auditar_nulos

import matplotlib.pyplot as plt

wellness = load_wellness()

print(f"{len(wellness)} dias | {wellness['date'].min().date()} a {wellness['date'].max().date()}")
wellness.head()

## 1. O que está preenchido

Antes de qualquer média, ver quanto de cada coluna existe. Colunas de sono faltam nas noites em que o
relógio saiu do pulso; `spo2` e `vo2max` têm comportamento próprio (ver `00_overview`).

In [ ]:
auditar_nulos(wellness)

## 2. Séries diárias e tendência de 7 dias

A linha fina é o dia; a grossa é a média móvel de 7 dias. A média móvel é o que interessa: HRV e FC de
repouso variam muito de um dia para o outro por causa de álcool, refeição tarde, calor e horário de
medição — a leitura de tendência só aparece depois de suavizar.

A referência tracejada no primeiro painel são 7h30 de sono, marco de comparação, não meta prescrita.

In [ ]:
series = [
    ("sleep_hours", "Horas de sono", "#3498db", 7.5),
    ("hrv", "HRV (ms)", "#27ae60", None),
    ("resting_hr", "FC de repouso (bpm)", "#e74c3c", None),
    ("avg_stress", "Estresse médio", "#f39c12", None),
]

fig, axes = plt.subplots(len(series), 1, figsize=(13, 10), sharex=True)

for ax, (col, titulo, cor, referencia) in zip(axes, series):
    ax.plot(wellness["date"], wellness[col], color=cor, alpha=0.30, linewidth=1, label="diário")
    ax.plot(
        wellness["date"],
        wellness[col].rolling(7, min_periods=3).mean(),
        color=cor, linewidth=2, label="média 7 dias",
    )
    if referencia is not None:
        ax.axhline(referencia, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.set_ylabel(titulo)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", frameon=False, fontsize=8)

axes[0].set_title("Wellness diário e tendência de 7 dias")
plt.tight_layout()
plt.show()

## 3. Distribuições — o que é normal para este atleta

Referência externa ("HRV bom é acima de X") não vale muito: HRV depende de idade, método de medição e
indivíduo. O que serve de baseline é a **própria distribuição** — é ela que diz se o valor de hoje é
alto ou baixo para esta pessoa. A linha vermelha marca a média.

In [ ]:
cols = ["sleep_hours", "sleep_score", "hrv", "resting_hr"]

fig, axes = plt.subplots(1, len(cols), figsize=(15, 3.6))

for ax, col in zip(axes, cols):
    dados = wellness[col].dropna()
    ax.hist(dados, bins=15, color="#3498db", alpha=0.85, edgecolor="white")
    ax.axvline(dados.mean(), color="#e74c3c", linestyle="--", linewidth=1.3)
    ax.set_title(f"{col}\nmédia {dados.mean():.1f} · mediana {dados.median():.1f} · n={len(dados)}", fontsize=9)
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

wellness[cols].describe().round(1)

## 4. Estágios do sono, semana a semana

Horas médias por noite em cada estágio, agregadas por semana. Duas leituras: a altura total mostra se
o volume de sono caiu, e a composição mostra se a **qualidade** mudou — sono profundo encolhendo com o
total constante é um sinal diferente de simplesmente dormir menos.

In [ ]:
estagios = {
    "sleep_deep_seconds": ("Profundo", "#1f4e79"),
    "sleep_rem_seconds": ("REM", "#3498db"),
    "sleep_light_seconds": ("Leve", "#a9cce3"),
    "sleep_awake_seconds": ("Acordado", "#e74c3c"),
}

por_semana = wellness.groupby("semana")[list(estagios)].mean() / 3600

fig, ax = plt.subplots(figsize=(13, 5))
ax.stackplot(
    por_semana.index.astype(str),
    *[por_semana[col] for col in estagios],
    labels=[rotulo for rotulo, _ in estagios.values()],
    colors=[cor for _, cor in estagios.values()],
    alpha=0.85,
)
ax.set_xlabel("Semana (numeração global do projeto)")
ax.set_ylabel("Horas por noite (média da semana)")
ax.set_title("Composição do sono por semana")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.tight_layout()
plt.show()

## 5. Resumo semanal

A mesma série em tabela, para checar número em vez de olhar gráfico. `dias` avisa quando a semana está
incompleta — semana com 3 dias registrados tem média que não se compara com uma de 7.

In [ ]:
semanal = (
    wellness.groupby("semana")
    .agg(
        dias=("date", "count"),
        sono_h=("sleep_hours", "mean"),
        sono_score=("sleep_score", "mean"),
        hrv=("hrv", "mean"),
        fc_repouso=("resting_hr", "mean"),
        estresse=("avg_stress", "mean"),
        passos=("steps", "mean"),
    )
    .round(1)
)

semanal.tail(12)

## Observações

_A preencher a cada atualização — este é o registro que sobrevive à próxima rodada de dados._

- Janela: 2026-04-08 a 2026-08-04, 119 dias.
- `sleep_awake_seconds` falta em 19% das noites; as demais colunas de sono ficam abaixo de 9%.
- Preencher aqui: nível típico de HRV e FC de repouso, semanas atípicas e o que aconteceu nelas
  (viagem, doença, bloco forte de treino).